# Week 5 - Spark DataFrames

Data cleaning, transformation and aggregation on the Superstore dataset using PySpark. Answers to all 15 questions are below, theory ones are written out and the code ones are run on the actual dataset.

In [1]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("week5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.csv("data/Sample - Superstore.csv", header=True, inferSchema=True, escape='"')
print(df.count(), "rows")
df.printSchema()

9994 rows
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



## Q1. Limitations of MapReduce vs Spark

MapReduce writes intermediate results to disk after every map and reduce phase, so a job with multiple stages keeps hitting HDFS again and again. That makes it slow, especially for anything iterative. Other problems:

- every problem has to be forced into a map + reduce shape, even joins and filters that don't naturally fit
- high latency, it's batch only, no interactive querying
- iterative algorithms (ML, graph processing) reread the same input from disk every single pass
- the Java API is verbose, you end up needing extra tools like Hive or Pig on top

Spark fixes most of this. It keeps data in memory between stages, builds a DAG of operations instead of rigid map/reduce pairs, and gives one engine for SQL, streaming and ML. For typical workloads it ends up 10-100x faster.

## Q2. In-memory computing and iterative ML

An algorithm like gradient descent or k-means goes over the same training data many times. On MapReduce every iteration is a fresh job - read from HDFS, compute, write back to HDFS. The disk I/O dominates the runtime.

In Spark you load the dataset once and `.cache()` it. Every iteration after the first reads straight from RAM, so the cost per pass drops massively. Fault tolerance still works because Spark remembers the lineage of transformations and can recompute a lost partition instead of relying on disk copies. This is basically why MLlib exists on Spark and not on MapReduce.

## Q3. Dropping duplicates on specific columns

For a generic df with those columns:

```python
df.dropDuplicates(["user_id", "transaction_date"])
```

On Superstore the closest equivalent is one row per customer per order date:

In [2]:
dedup = df.dropDuplicates(["Customer ID", "Order Date"])
print("before:", df.count(), " after:", dedup.count())

before: 9994  after: 4992


## Q4. Filter West region, average sales per category

Asked form: `df_sales.filter(F.col("region") == "West").groupBy("product_category").agg(F.avg("sale_amount"))`

Same thing on our columns:

In [3]:
(df.filter(F.col("Region") == "West")
   .groupBy("Category")
   .agg(F.round(F.avg("Sales"), 2).alias("avg_sales"))
   .orderBy(F.desc("avg_sales"))
   .show())

+---------------+---------+
|       Category|avg_sales|
+---------------+---------+
|     Technology|   420.69|
|      Furniture|    357.3|
|Office Supplies|   116.42|
+---------------+---------+



## Q5. na.drop() vs na.fill()

`.na.drop()` throws away rows containing nulls, `.na.fill()` keeps the rows and replaces the nulls with a value you choose. Drop when the row is useless without the value, fill when you can substitute something sensible.

Superstore has no nulls (checked below in Q9), so here's a small frame to demo filling a `status` column with 'Unknown':

In [4]:
rows = [
    (1, "delivered", "a@mail.com", "arjun", 120.0),
    (2, None, "b@mail.com", "priya", None),
    (3, "returned", None, "rahul", 80.5),
    (4, None, "d@mail.com", "", 40.0),
    (5, "shipped", "e@mail.com", "sneha", None),
]
orders = spark.createDataFrame(rows, ["order_id", "status", "email", "username", "price"])
orders.na.fill({"status": "Unknown"}).show()

+--------+---------+----------+--------+-----+
|order_id|   status|     email|username|price|
+--------+---------+----------+--------+-----+
|       1|delivered|a@mail.com|   arjun|120.0|
|       2|  Unknown|b@mail.com|   priya| NULL|
|       3| returned|      NULL|   rahul| 80.5|
|       4|  Unknown|d@mail.com|        | 40.0|
|       5|  shipped|e@mail.com|   sneha| NULL|
+--------+---------+----------+--------+-----+



## Q6. Cities with more than 100 records

Group, count, then filter on the aggregated count:

In [5]:
(df.groupBy("City")
   .count()
   .filter(F.col("count") > 100)
   .orderBy(F.desc("count"))
   .show())

+-------------+-----+
|         City|count|
+-------------+-----+
|New York City|  915|
|  Los Angeles|  747|
| Philadelphia|  537|
|San Francisco|  510|
|      Seattle|  428|
|      Houston|  377|
|      Chicago|  314|
|     Columbus|  222|
|    San Diego|  170|
|  Springfield|  163|
|       Dallas|  157|
| Jacksonville|  125|
|      Detroit|  115|
+-------------+-----+



## Q7. Immutability and cleaning steps

A DataFrame can never be modified in place. Dropping a column or renaming it doesn't touch the original, it returns a brand new DataFrame, so you always reassign:

```python
df = df.drop("junk_col").withColumnRenamed("old", "new")
```

In practice cleaning becomes a chain of transformations where each step produces a new frame. Nothing actually executes until an action like `.show()` or `.count()` triggers it, Spark just keeps building the plan. The upside is the raw df is still there untouched if a cleaning step goes wrong.

## Q8. Range filter + equality filter

Asked form:

```python
df.filter((F.col("age").between(18, 30)) & (F.col("subscription") == "Premium"))
```

No age column here, so same pattern using Discount and Segment:

In [6]:
corp = df.filter((F.col("Discount").between(0.1, 0.3)) & (F.col("Segment") == "Corporate"))
print(corp.count(), "rows")
corp.select("Customer Name", "Category", "Sales", "Discount").show(5)

1228 rows


+--------------+---------------+--------+--------+
| Customer Name|       Category|   Sales|Discount|
+--------------+---------------+--------+--------+
|     Gene Hale|     Technology|1097.544|     0.2|
|Linda Cazamias|     Technology| 147.168|     0.2|
|    Erin Smith|Office Supplies|  95.616|     0.2|
| Brendan Sweed|Office Supplies|1113.024|     0.2|
| Brendan Sweed|     Technology| 167.968|     0.2|
+--------------+---------------+--------+--------+
only showing top 5 rows


## Q9. Why handle nulls before aggregating

Spark quietly skips nulls in `sum()` and `avg()`. That sounds fine but `avg` divides by the count of non-null values, not total rows, so a column with lots of nulls gives a misleading average. Two different aggregations can also end up computed over different row counts. Handling nulls first (fill or drop, deliberately) means you know exactly what the numbers are based on instead of getting silently shifted results.

Checking Superstore for nulls:

In [7]:
df.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns]).show(vertical=True)

-RECORD 0------------
 Row ID        | 0   
 Order ID      | 0   
 Order Date    | 0   
 Ship Date     | 0   
 Ship Mode     | 0   
 Customer ID   | 0   
 Customer Name | 0   
 Segment       | 0   
 Country       | 0   
 City          | 0   
 State         | 0   
 Postal Code   | 0   
 Region        | 0   
 Product ID    | 0   
 Category      | 0   
 Sub-Category  | 0   
 Product Name  | 0   
 Sales         | 0   
 Quantity      | 0   
 Discount      | 0   
 Profit        | 0   



## Q10. Cast to timestamp + rename

Asked form:

```python
df.withColumn("raw_timestamp", F.col("raw_timestamp").cast("timestamp")) \
  .withColumnRenamed("raw_timestamp", "event_time")
```

Order Date comes in as a string like 11/8/2016, so it needs `to_timestamp` with the format:

In [8]:
fixed = (df.withColumn("Order Date", F.to_timestamp("Order Date", "M/d/yyyy"))
           .withColumnRenamed("Order Date", "order_ts"))
fixed.select("order_ts", "Ship Date").show(3)
fixed.select("order_ts").printSchema()

+-------------------+----------+
|           order_ts| Ship Date|
+-------------------+----------+
|2016-11-08 00:00:00|11/11/2016|
|2016-11-08 00:00:00|11/11/2016|
|2016-06-12 00:00:00| 6/16/2016|
+-------------------+----------+
only showing top 3 rows
root
 |-- order_ts: timestamp (nullable = true)



## Q11. The shuffle and wide transformations

A `groupBy` needs all rows with the same key on the same executor before it can aggregate them. Since matching keys are scattered across partitions, Spark has to redistribute the data - each task writes its rows into buckets by key, they go over the network, and the next stage reads its bucket from every task. That whole exchange is the shuffle.

It's called a wide transformation because one output partition depends on many input partitions (unlike `filter` or `select` where each output partition comes from exactly one input partition). Wide transformations force a stage boundary and involve disk + network, which is why they're the expensive part of a Spark job.

## Q12. Remove rows with null email OR empty username

Removing those rows means keeping the ones where email is present AND username is non-empty:

In [9]:
valid = orders.filter(F.col("email").isNotNull() & (F.col("username") != ""))
valid.show()

+--------+---------+----------+--------+-----+
|order_id|   status|     email|username|price|
+--------+---------+----------+--------+-----+
|       1|delivered|a@mail.com|   arjun|120.0|
|       2|     NULL|b@mail.com|   priya| NULL|
|       5|  shipped|e@mail.com|   sneha| NULL|
+--------+---------+----------+--------+-----+



## Q13. Multiple statistics with .agg()

Asked form with a price column:

```python
df.agg(F.min("price"), F.max("price"), F.mean("price"))
```

On Sales:

In [10]:
df.agg(
    F.min("Sales").alias("min_sale"),
    F.max("Sales").alias("max_sale"),
    F.round(F.mean("Sales"), 2).alias("mean_sale"),
).show()

+--------+--------+---------+
|min_sale|max_sale|mean_sale|
+--------+--------+---------+
|   0.444|22638.48|   229.86|
+--------+--------+---------+



## Q14. Risk of inferSchema=true with messy data

inferSchema only samples the data and picks whatever type fits. With inconsistent date formats the column just falls back to string, or worse, values that don't parse become nulls without any warning, so you lose data silently.

This dataset actually demonstrated it. Some product names contain quote characters, and on the first read the parser broke those rows - Sales got inferred as string because a fragment of a product name leaked into it. Adding `escape='"'` fixed the parsing and Sales came back as double. The safer habit for anything messy is reading dates as strings and converting explicitly with `to_timestamp`, or defining the schema yourself.

## Q15. Full pipeline: dedupe -> fill nulls -> revenue per group

Asked form:

```python
(df.dropDuplicates()
   .na.fill({"price": 0})
   .groupBy("store_id")
   .agg(F.sum(F.col("price") * F.col("quantity")).alias("total_revenue")))
```

On Superstore, deduping on Order ID + Product ID, filling Sales, revenue per state:

In [11]:
revenue = (df.dropDuplicates(["Order ID", "Product ID"])
             .na.fill({"Sales": 0})
             .groupBy("State")
             .agg(F.round(F.sum("Sales"), 2).alias("total_revenue"))
             .orderBy(F.desc("total_revenue")))
revenue.show(10)

revenue.toPandas().to_csv("output/state_revenue.csv", index=False)

+------------+-------------+
|       State|total_revenue|
+------------+-------------+
|  California|    457687.63|
|    New York|    310827.15|
|       Texas|    170188.05|
|  Washington|    138641.27|
|Pennsylvania|    116511.91|
|     Florida|     89473.71|
|    Illinois|      80166.1|
|        Ohio|     77976.76|
|    Michigan|     76269.61|
|    Virginia|     70350.34|
+------------+-------------+
only showing top 10 rows


## Insights

- California ($457k) and New York ($310k) generate the most revenue by a wide margin, the top 4 states alone cover a huge chunk of total sales (~$2.29M overall).
- In the West, Technology has the highest average order value (~$421) while Office Supplies is much lower (~$116) - lots of small purchases vs fewer big ones.
- New York City (915), Los Angeles (747) and Philadelphia (537) are the busiest cities, matching the state revenue picture.
- The dataset itself was clean null-wise, but the CSV parsing issue (quotes inside product names) was a good reminder that schema problems can hide in the file format, not just the values. Fixing the read options mattered more than any null handling here.

In [12]:
spark.stop()